In [2]:
import sys, sklearn, skl2onnx, onnx, onnxruntime as ort
print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("skl2onnx:", skl2onnx.__version__)
print("onnx:", onnx.__version__)
print("onnxruntime:", ort.__version__)
print("providers:", ort.get_available_providers())  # en az 'CPUExecutionProvider' görünmeli


Python: 3.13.3
scikit-learn: 1.6.1
skl2onnx: 1.19.1
onnx: 1.18.0
onnxruntime: 1.22.1
providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


In [5]:
# === IsolationForest eğitimi + ONNX'e çevirme (tek hücre) ===
import os, json, numpy as np
from sklearn.ensemble import IsolationForest
from skl2onnx import to_onnx

# 1) Özellik listesi (Rust tarafıyla aynı sırada tutacağız)
FEATURES = [
    "domain_len","entropy","labels","is_txt",
    "nx_ratio","qps_client","unique_sub_ratio"
]

# 2) Sentetik "iyi/kötü" karışımı (gözetimsiz eğitim için yeterli)
rng = np.random.default_rng(42)
good = np.column_stack([
    rng.normal(22,5,1000).clip(5,63),          # domain_len
    rng.normal(3.2,0.35,1000).clip(1.5,4.2),   # entropy
    rng.integers(1,10,1000),                   # labels
    rng.binomial(1,0.05,1000),                 # is_txt
    rng.beta(1,8,1000),                        # nx_ratio
    rng.gamma(2.0,0.15,1000),                  # qps_client
    rng.beta(1,5,1000)                         # unique_sub_ratio
]).astype("float32")

bad = np.column_stack([
    rng.integers(45,63,60),                    # uzun
    rng.uniform(3.9,5.0,60),                   # yüksek entropi
    rng.integers(4,10,60),
    rng.binomial(1,0.8,60),                    # TXT ağırlıklı
    rng.uniform(0.6,1.0,60),                   # NX yüksek
    rng.uniform(8,15,60),                      # QPS yüksek
    rng.uniform(0.6,1.0,60)                    # benzersiz sub yüksek
]).astype("float32")

X = np.vstack([good, bad])

# 3) IsolationForest (skor düştükçe daha anormal; decision_function<0 outlier)
iforest = IsolationForest(
    n_estimators=200, max_samples=256,
    contamination="auto", random_state=42, n_jobs=-1
).fit(X)

# 4) ONNX'e çevir (score_samples çıktısını da ekle)
onnx_model = to_onnx(
    iforest, X[:5],
    options={id(iforest): {"score_samples": True}},
    target_opset={'': 21, 'ai.onnx.ml': 3}   # <-- kritik düzeltme
)


os.makedirs("models", exist_ok=True)
with open("models/iforest.onnx","wb") as f:
    f.write(onnx_model.SerializeToString())
with open("models/iforest.features.json","w") as f:
    json.dump(FEATURES, f)

print("Saved:", "models/iforest.onnx")
print("Saved:", "models/iforest.features.json")


Saved: models/iforest.onnx
Saved: models/iforest.features.json
